# Can you trace a product claim?
### Three practical data audits with a padel-racket dataset

A product catalogue can contain a number without showing where it came from. It can contain a source link without proving the claim. And it can be freshly exported while carrying older observations.

This notebook asks three questions:

1. **Traceability:** which recorded specification objects include usable provenance metadata?
2. **Missingness:** what is absent from the records, and how does that differ from the publisher's five-input completeness summary?
3. **Time and identity:** which dates and model identifiers can support a reproducible comparison?

The aim is a reusable audit, not a ranking of rackets, brands, or shops. No prediction, performance label, injury outcome, or buying recommendation is produced.

**Disclosure:** this notebook is published by the PadelTrue dataset publisher, whose website earns a commission from some shop links. It is an original tutorial prepared and checked with AI assistants; the code was executed against public snapshots. The analysis uses public data; no visitor or conversation data is included. Links below identify the source and its limitations; none are shop links.


## Before running

Attach the existing Kaggle dataset **[Padel racket specifications with sources](https://www.kaggle.com/datasets/odedkovach/padel-racket-specifications-with-sources)**. The JSON is necessary: the CSV does not preserve all field-level sources.

The notebook automatically finds `rackets.json` in the attached Kaggle input. Offline/local alternative: place a frozen public `rackets.json` beside this notebook, or set the `PADEL_DATA_PATH` environment variable. It makes **no network requests** and does not require Kaggle secrets, an LLM, or paid compute.

Counts below are computed from the attached version. A dataset snapshot is not the entire racket market. Missing from our records does not mean never published elsewhere.


In [1]:
import csv
import hashlib
import json
import os
from collections import Counter, defaultdict
from datetime import date, timedelta
from html import escape
from pathlib import Path
from urllib.parse import urlsplit
from IPython.display import HTML, Markdown, display

# Fixed, visible reference date makes the frozen example reproducible.
# Change it deliberately when asking a different date-specific question.
AS_OF = date(2026, 10, 14)
OUTPUT_DIR = Path.cwd() / 'audit-output'
OUTPUT_DIR.mkdir(exist_ok=True)


def valid_http_url(value):
    if not isinstance(value, str) or value != value.strip():
        return False
    try:
        parts = urlsplit(value)
        return parts.scheme in {'http', 'https'} and bool(parts.hostname)
    except ValueError:
        return False


def iso_day(value):
    if not isinstance(value, str) or len(value) != 10:
        return None
    try:
        return date.fromisoformat(value)
    except ValueError:
        return None


def has_value(value):
    # False and 0 are valid values, not missing data.
    return value is not None and value != '' and value != [] and value != {}


def source_type(spec):
    value = spec.get('sourceType')
    return value if value in {'manufacturer', 'retailer'} else 'unclassified'


def show_table(headers, rows, caption=''):
    head = ''.join('<th style="text-align:left;padding:9px;border-bottom:2px solid #183b56">'
                   + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td style="padding:9px;border-bottom:1px solid #ddd">'
                   + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<div style="overflow-x:auto"><table style="border-collapse:collapse;width:100%">'
                 + '<caption style="text-align:left;padding:8px 0">' + escape(caption)
                 + '</caption><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table></div>'))


def read_candidate(path):
    try:
        value = json.loads(path.read_text(encoding='utf-8'))
        if isinstance(value, dict) and isinstance(value.get('rackets'), list) and value['rackets']:
            if 'PadelTrue' in str(value.get('name', '')) and all(isinstance(r, dict) for r in value['rackets']):
                return value
    except (OSError, ValueError):
        pass
    return None


def choose_input():
    override = os.environ.get('PADEL_DATA_PATH')
    if override:
        p = Path(override)
        data = read_candidate(p)
        if data is None:
            raise ValueError('PADEL_DATA_PATH must point to a PadelTrue JSON export with a nonempty rackets list.')
        return p, data, 'explicit frozen input'
    root = Path(os.environ.get('KAGGLE_INPUT_ROOT', '/kaggle/input'))
    candidates = sorted(root.rglob('rackets.json')) if root.is_dir() else []
    matches = [(p, read_candidate(p)) for p in candidates]
    matches = [(p, d) for p, d in matches if d is not None]
    if matches:
        hashes = {hashlib.sha256(p.read_bytes()).hexdigest() for p, d in matches}
        if len(hashes) != 1:
            raise ValueError('More than one different PadelTrue snapshot is attached. Set PADEL_DATA_PATH explicitly.')
        return matches[0][0], matches[0][1], 'attached Kaggle dataset'
    local = Path('rackets.json')
    data = read_candidate(local)
    if data is not None:
        return local, data, 'local frozen snapshot'
    raise FileNotFoundError('Attach the existing PadelTrue Kaggle dataset, or place rackets.json beside the notebook.')

input_path, dataset, input_mode = choose_input()
raw_hash = hashlib.sha256(input_path.read_bytes()).hexdigest()
rackets = dataset['rackets']
N = len(rackets)
show_table(['Property', 'Observed value'], [
    ['Input mode', input_mode], ['File', input_path.name],
    ['Records actually read', N], ['Declared count', dataset.get('count', 'not recorded')],
    ['Export date (not a verification date)', dataset.get('built', 'not recorded')],
    ['Analysis reference date', AS_OF.isoformat()],
    ['Declared data license', dataset.get('license', 'not recorded')],
    ['SHA-256 of exact input bytes', raw_hash],
], 'The fingerprint identifies this run, even when a later dataset version changes the counts.')


Property,Observed value
Input mode,local frozen snapshot
File,rackets.json
Records actually read,283
Declared count,283
Export date (not a verification date),2026-10-14
Analysis reference date,2026-10-14
Declared data license,CC BY 4.0
SHA-256 of exact input bytes,5a6579c171ba7da4d6ad173328b246e3f2334948269ee7d2d9be6ee4bc90fde1


## 1. Can each recorded specification be traced?

We flatten **recorded specification objects**, not every fact about a racket. Each object has at most one recorded source URL here; several fields citing the same page are not independent corroboration.

These checks test metadata structure only. A syntactically valid HTTP(S) URL might be unavailable, refer to another model, or contradict the stored value. A `manufacturer` label is the publisher's classification, not our independent verification of the domain. Unrecognized or absent classifications remain **unclassified**.


In [2]:
claims = []
for racket in rackets:
    specs = racket.get('specs')
    if not isinstance(specs, dict):
        continue
    for field, spec in specs.items():
        if not isinstance(spec, dict):
            continue
        url = spec.get('source')
        good_url = valid_http_url(url)
        claims.append({
            'slug': racket.get('slug'), 'field': field,
            'source_type': source_type(spec), 'url_present': has_value(url),
            'http_url': good_url,
            'host': urlsplit(url).hostname.lower() if good_url else None,
            'wording_present': isinstance(spec.get('verbatim'), str) and bool(spec['verbatim'].strip()),
            'checked_on': iso_day(spec.get('checkedOn')),
        })

by_field = defaultdict(list)
for claim in claims:
    by_field[claim['field']].append(claim)
trace_rows = []
for field, group in sorted(by_field.items()):
    trace_rows.append([field, len(group), sum(c['http_url'] for c in group),
                       sum(c['wording_present'] for c in group),
                       sum(c['source_type'] == 'unclassified' for c in group)])
show_table(['Field', 'Objects', 'HTTP(S) source URL', 'Recorded wording', 'Unclassified source'], trace_rows,
           'Denominator for each row: recorded objects of that field, not all rackets.')
source_counts = Counter(c['source_type'] for c in claims)
show_table(['Recorded category', 'Specification objects'],
           [[kind, source_counts[kind]] for kind in ['manufacturer', 'retailer', 'unclassified']],
           'These are recorded source categories, not independently verified authority scores.')

host_fields = Counter(c['host'] for c in claims if c['host'])
host_models = defaultdict(set)
for c in claims:
    if c['host']:
        host_models[c['host']].add(c['slug'])
show_table(['Source host', 'Objects citing it', 'Distinct model IDs'],
           [[host, count, len(host_models[host])] for host, count in host_fields.most_common(5)],
           'Top five recorded hosts. Repeated claims can share one source page.')


Field,Objects,HTTP(S) source URL,Recorded wording,Unclassified source
balance,220,220,220,0
core,267,267,267,0
frame,137,137,137,0
price,283,283,283,0
roughSurface,210,210,210,0
shape,267,267,267,0
surface,257,257,257,0
thicknessMm,163,163,163,0
weight,240,240,240,0


Recorded category,Specification objects
manufacturer,1301
retailer,743
unclassified,0


Source host,Objects citing it,Distinct model IDs
www.padelnuestro.com,452,67
noxsport.com,345,43
allforpadel.com,246,36
www.bullpadel.com,222,28
www.babolat.com,120,14


**Exercise 1.** Why is the count of specification objects not the count of independent sources? Modify the next cell to count distinct source URLs across the whole export, then compare the two totals. Pick one row and inspect its original source manually before quoting it.


In [3]:
# Worked check: a distinct page is still not necessarily an independent publisher.
distinct_source_pages = {
    spec['source']
    for racket in rackets
    for spec in (racket.get('specs') or {}).values()
    if isinstance(spec, dict) and valid_http_url(spec.get('source'))
}
print(f'{len(claims):,} recorded specification objects; {len(distinct_source_pages):,} distinct source URLs.')
print('No source page was fetched or fact-checked by this notebook.')


2,044 recorded specification objects; 310 distinct source URLs.
No source page was fetched or fact-checked by this notebook.


## 2. Which information is missing from these records?

Two denominators answer different questions:

- **Object coverage:** does a record contain a nonempty specification object for this field?
- **Declared model-input coverage:** does the export say its rating calculation had five usable inputs?

A material name may be present while the calculation cannot normalize it. Therefore an object being present is not equivalent to a usable model input. We inspect the input-coverage metadata without reading or evaluating any calculated ratings or profiles.

A blank value, an absent key and a malformed object are reported separately below. `False` (for example a surface flag) and `0` are not converted to missing values.


In [4]:
EXPECTED_FIELDS = ['shape', 'weight', 'balance', 'core', 'surface', 'frame', 'thicknessMm', 'roughSurface', 'price']
object_rows = []
for field in EXPECTED_FIELDS:
    counts = Counter()
    for racket in rackets:
        specs = racket.get('specs')
        if not isinstance(specs, dict) or field not in specs:
            counts['absent'] += 1
        elif specs[field] is None:
            counts['null'] += 1
        elif not isinstance(specs[field], dict):
            counts['malformed'] += 1
        elif len(specs[field]) == 0:
            counts['empty object'] += 1
        else:
            counts['object present'] += 1
    assert sum(counts.values()) == N
    object_rows.append([field, counts['object present'], counts['absent'], counts['null'],
                        counts['empty object'], counts['malformed']])
show_table(['Field', 'Object present', 'Key absent', 'Null', 'Empty object', 'Other type'], object_rows,
           f'Each row partitions all {N:,} racket records. Presence does not certify a usable value.')

VALID_INPUTS = {'shape', 'balance', 'core', 'face', 'weight'}
input_coverage = Counter()
input_issues = []
for racket in rackets:
    published = racket.get('inputsPublished')
    missing = racket.get('inputsMissing')
    numeric_ok = type(published) is int and 0 <= published <= 5
    input_coverage[published if numeric_ok else 'unknown / invalid'] += 1
    list_ok = isinstance(missing, list) and all(isinstance(x, str) and x in VALID_INPUTS for x in missing)
    list_ok = list_ok and len(missing) == len(set(missing))
    if not (numeric_ok and list_ok and published + len(missing) == 5):
        input_issues.append({'slug': racket.get('slug'), 'issue': 'missing or inconsistent five-input metadata'})
show_table(['Declared usable inputs', 'Records'],
           [[i, input_coverage[i]] for i in range(6)] + [['unknown / invalid', input_coverage['unknown / invalid']]],
           'A description of the publisher calculation inputs, not equipment performance.')
print(f'Five-input metadata needing inspection: {len(input_issues):,} records.')

missing_input_counts = Counter(x for r in rackets for x in (r.get('inputsMissing') or []) if x in VALID_INPUTS)
show_table(['Declared missing input', 'Records naming it'], sorted(missing_input_counts.items()),
           'One record can name several missing inputs; do not sum this column to count unique rackets.')


Field,Object present,Key absent,Null,Empty object,Other type
shape,267,16,0,0,0
weight,240,43,0,0,0
balance,220,63,0,0,0
core,267,16,0,0,0
surface,257,26,0,0,0
frame,137,146,0,0,0
thicknessMm,163,120,0,0,0
roughSurface,210,73,0,0,0
price,283,0,0,0,0


Declared usable inputs,Records
0,0
1,0
2,0
3,76
4,113
5,94
unknown / invalid,0


Five-input metadata needing inspection: 0 records.


Declared missing input,Records naming it
balance,88
core,72
face,46
shape,16
weight,43


**Exercise 2.** How many records have a `core` object yet list `core` among the missing calculation inputs? Before calling that an error, read a few source records. A published foam/material name and a usable normalized category are different facts.


In [5]:
core_mismatch = [r for r in rackets
                 if isinstance((r.get('specs') or {}).get('core'), dict)
                 and 'core' in (r.get('inputsMissing') or [])]
print(f'{len(core_mismatch):,} records have a core object but declare core missing from calculation inputs.')
show_table(['Model ID', 'Core material recorded', 'Normalized core category'],
           [[r.get('slug'), r['specs']['core'].get('material') or 'not recorded',
             r['specs']['core'].get('value') if has_value(r['specs']['core'].get('value')) else 'not recorded']
            for r in core_mismatch[:5]], 'First five examples, sorted as in the input. This is an inspection queue, not a verdict.')


56 records have a core object but declare core missing from calculation inputs.


Model ID,Core material recorded,Normalized core category
babolat-air-veron-2-6,Black EVA,not recorded
babolat-air-vertuo-2-6,Black EVA,not recorded
babolat-air-viper-2-6,X-EVA,not recorded
babolat-counter-veron-2-6,Black EVA,not recorded
babolat-counter-viper-2-6,X-EVA,not recorded


## 3. Is a fresh export the same as a fresh observation?

No. The root `built` date identifies an export. A specification's `checkedOn`, when present, identifies a recorded observation day. We never fill missing observation dates with the export date.

The following age buckets are relative to the visible reference date in the setup cell. They are calendar-day descriptions, **not stock status, an SLA, or proof that a price is still available**. An undated specification may be correct; its freshness is simply not established here.

Model years also remain explicit. An unknown year is not the current year. Preserve exact model IDs and variants when joining this dataset to another source.


In [6]:
def observation_bucket(day):
    if day is None:
        return 'unknown / absent date'
    age = (AS_OF - day).days
    if age < 0:
        return 'after reference date'
    if age == 0:
        return 'same calendar day'
    if age <= 7:
        return '1 to 7 days before'
    return '8+ days before'

BUCKETS = ['same calendar day', '1 to 7 days before', '8+ days before', 'after reference date', 'unknown / absent date']
date_rows = []
for field, group in sorted(by_field.items()):
    buckets = Counter(observation_bucket(c['checked_on']) for c in group)
    date_rows.append([field] + [buckets[b] for b in BUCKETS])
show_table(['Field'] + BUCKETS, date_rows, f'Observation metadata relative to {AS_OF.isoformat()}; missing dates remain unknown.')

slug_counts = Counter(r.get('slug') for r in rackets if has_value(r.get('slug')))
year_unknown = sum(r.get('year') is None for r in rackets)
year_invalid = sum(r.get('year') is not None and type(r.get('year')) is not int for r in rackets)
identity_summary = [
    ['Declared count differs from actual length', int(dataset.get('count') != N)],
    ['Missing/blank model ID', sum(not has_value(r.get('slug')) for r in rackets)],
    ['Model IDs appearing more than once', sum(n > 1 for n in slug_counts.values())],
    ['Unknown year (null or absent)', year_unknown],
    ['Non-integer, non-null year', year_invalid],
    ['Missing/invalid publisher model-page URL', sum(not valid_http_url(r.get('url')) for r in rackets)],
]
show_table(['Identity check', 'Count'], identity_summary,
           'Unknown year is retained as unknown. A count of zero means only that this structural check found none.')


Field,same calendar day,1 to 7 days before,8+ days before,after reference date,unknown / absent date
balance,0,0,0,0,220
core,0,0,0,0,267
frame,0,0,0,0,137
price,2,244,37,0,0
roughSurface,0,0,1,0,209
shape,0,0,0,0,267
surface,0,0,0,0,257
thicknessMm,0,0,0,0,163
weight,0,1,0,0,239


Identity check,Count
Declared count differs from actual length,0
Missing/blank model ID,0
Model IDs appearing more than once,0
Unknown year (null or absent),70
"Non-integer, non-null year",0
Missing/invalid publisher model-page URL,0


**Exercise 3.** Pretend a downstream application sets every missing `checkedOn` to the export date. How many objects acquire an unsupported observation date? Run the cell below, then explain why calling those objects “verified today” would be misleading.


In [7]:
unsupported_date_fills = sum(c['checked_on'] is None for c in claims)
print(f'That fill would assign an unsupported observation date to {unsupported_date_fills:,} specification objects.')
print('Keep the export date and observation date in separate columns.')


That fill would assign an unsupported observation date to 1,759 specification objects.
Keep the export date and observation date in separate columns.


## Export a small, reproducible audit

The report records its input fingerprint, source metadata, reference date and definitions. The CSVs contain aggregate audit results, not a second copy of the catalogue. They do not include user data, shopping commissions, source-page quotations or calculated racket performance labels.


In [8]:
# Small edge-case checks protect the audit's interpretation of unknown and false.
assert has_value(False) and has_value(0)
assert not has_value(None) and not has_value('')
assert valid_http_url('https://example.org/item')
assert not valid_http_url('javascript:alert(1)')
assert not valid_http_url('https:///item')
assert iso_day('2026-02-30') is None
assert source_type({}) == 'unclassified'
assert observation_bucket(None) == 'unknown / absent date'
assert observation_bucket(AS_OF + timedelta(days=1)) == 'after reference date'

summary = {
    'title': 'Product-data provenance and missingness audit',
    'input_sha256': raw_hash,
    'input_name': input_path.name,
    'input_mode': input_mode,
    'dataset_export_date': dataset.get('built'),
    'data_license_as_recorded': dataset.get('license'),
    'analysis_reference_date': AS_OF.isoformat(),
    'records': N,
    'specification_objects': len(claims),
    'distinct_source_urls': len(distinct_source_pages),
    'source_categories_as_recorded': dict(source_counts),
    'objects_without_valid_http_source': sum(not c['http_url'] for c in claims),
    'objects_without_observation_day': unsupported_date_fills,
    'unknown_model_years': year_unknown,
    'input_metadata_issues': len(input_issues),
    'source': 'https://padeltrue.com/data',
    'methodology': 'https://padeltrue.com/methodology',
    'limits': [
        'Metadata audit only; linked pages were not fetched or independently fact-checked.',
        'Missingness concerns this collection, not the whole market.',
        'Source categories are recorded classifications, not verified authority.',
        'No performance, comfort, injury, sales or stock inference is made.',
        'Export date never substitutes for a missing observation date.'
    ]
}
(OUTPUT_DIR / 'audit-summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
for filename, headers, rows in [
    ('provenance-by-field.csv', ['field', 'objects', 'http_source_url', 'recorded_wording', 'unclassified_source'], trace_rows),
    ('object-coverage.csv', ['field', 'object_present', 'key_absent', 'null', 'empty_object', 'other_type'], object_rows),
    ('observation-days.csv', ['field'] + BUCKETS, date_rows),
    ('identity-checks.csv', ['check', 'count'], identity_summary),
]:
    with (OUTPUT_DIR / filename).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle)
        writer.writerow(headers)
        writer.writerows(rows)
print('Saved audit-summary.json and four aggregate CSVs in audit-output/.')
print('Edge-case checks passed. A successful metadata audit is not a factual product verification.')


Saved audit-summary.json and four aggregate CSVs in audit-output/.
Edge-case checks passed. A successful metadata audit is not a factual product verification.


## What you can reuse

Use these checks on another product catalogue: separate field existence from usable values, count source pages separately from claims, and preserve unknown dates. When adapting the code, define your own field schema instead of assuming this one applies universally.

**A good next investigation:** choose a small random sample of source pages, record access time and exact model/year, and manually compare the original source with the stored field. Report unavailable pages separately from contradictions. That would extend this structural audit with new evidence; this notebook has not done it.

Sources and definitions:

- [PadelTrue open-data explanation](https://padeltrue.com/data)
- [Public dataset and data dictionary](https://github.com/odedkovach/padel-racket-data)
- [PadelTrue methodology and limitations](https://padeltrue.com/methodology)
- [Original Kaggle dataset](https://www.kaggle.com/datasets/odedkovach/padel-racket-specifications-with-sources)

Credit the dataset publisher under the attached export's stated license and retain the exact input fingerprint when sharing findings. A publisher link is provenance, not proof of independent validation.
